# Task 1

In [56]:
import pandas as pa
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

iris_data = pa.read_csv("iris_data.csv", delimiter=";").drop(columns=["row_no", "id"])
iris_data["label"] = iris_data["label"].map({'Iris-virginica' : 0, 'Iris-versicolor': 1, 'Iris-setosa': 2} )
iris_data

,label,a1,a2,a3,a4
0,0,5.9,3.0,5.1,1.8
1,0,6.2,3.4,5.4,2.3
2,0,6.5,3.0,5.2,2.0
3,0,6.3,2.5,5.0,1.9
4,0,6.7,3.0,5.2,2.3
...,...,...,...,...,...
145,2,5.0,3.6,1.4,0.2
146,2,4.6,3.1,1.5,0.2
147,2,4.7,3.2,1.3,0.2
148,2,4.9,3.0,1.4,0.2


In [57]:
x = iris_data.drop(columns="label")
y = iris_data["label"]
model = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=3))])
model.fit(x,y)
model.score(x, y)

0.9533333333333334

In [58]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
model_with_split = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=3))])
model_with_split.fit(x_train, y_train)
print("Score on train set: ", model_with_split.score(x_test, y_test))


Score on train set:  0.9666666666666667


The model that tests the accuracy on unseen data was lower, but when stratifying y i.e. splitting the dataset so that the dataset is better distributed and has more propotinal labels it got better accuarcy

In [59]:
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.4, random_state=42, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
model_with_val_split = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=3))])
results = []
for k in range(1, 31, 2):
    knn_model = Pipeline([("scaler", StandardScaler()),
                      ("knn", KNeighborsClassifier(n_neighbors=k))])
    knn_model.fit(x_train, y_train)                      # train ONLY on training set
    train_acc = knn_model.score(x_train, y_train)
    val_acc = knn_model.score(x_val, y_val)
    results.append((k, train_acc, val_acc, knn_model))
    print(f"k={k:2d}  train={train_acc:.3f}  val={val_acc:.3f}")
best_k, best_train, best_val, best_model = max(results, key=lambda r: r[2])
test_acc = best_model.score(x_test, y_test)
print(f"\nBest k = {best_k}")
print(f"Train: {best_train:.3f}  Validation: {best_val:.3f}  Test: {test_acc:.3f}")

k= 1  train=1.000  val=0.900
k= 3  train=0.967  val=0.933
k= 5  train=0.956  val=0.933
k= 7  train=0.967  val=0.933
k= 9  train=0.967  val=0.933
k=11  train=0.967  val=0.933
k=13  train=0.956  val=0.933
k=15  train=0.944  val=0.933
k=17  train=0.922  val=0.933
k=19  train=0.922  val=0.933
k=21  train=0.911  val=0.933
k=23  train=0.911  val=0.900
k=25  train=0.900  val=0.900
k=27  train=0.911  val=0.867
k=29  train=0.900  val=0.833

Best k = 3
Train: 0.967  Validation: 0.933  Test: 0.933


In [60]:
iris_data

,label,a1,a2,a3,a4
0,0,5.9,3.0,5.1,1.8
1,0,6.2,3.4,5.4,2.3
2,0,6.5,3.0,5.2,2.0
3,0,6.3,2.5,5.0,1.9
4,0,6.7,3.0,5.2,2.3
...,...,...,...,...,...
145,2,5.0,3.6,1.4,0.2
146,2,4.6,3.1,1.5,0.2
147,2,4.7,3.2,1.3,0.2
148,2,4.9,3.0,1.4,0.2


In [61]:
iris_data.columns

Index(['label', 'a1', 'a2', 'a3', 'a4'], dtype='str')

In [62]:
import pandas as pd
import plotly.express as px


cols = iris_data.columns[:3]
fig = px.scatter_3d(iris_data, x=cols[0], y=cols[1], z=cols[2], color="label", opacity=0.8)
fig.update_traces(marker_size=4)
fig.show()